# OVERVIEW OF THE STUDY

## Parameters to sweep

Parameter Range

Material A refractive indexnA∈[1.5,3.0]n_A \in [1.5, 3.0]nA​∈[1.5,3.0]

Material B refractive indexnB=1.5n_B = 1.5nB​=1.5

Extinction (both A & B)k∈[0,0.2]k \in [0, 0.2]k∈[0,0.2]

Design wavelengthvisible (e.g. 380–780 nm)

Number of layers10 → 100

Collection half‑angle10° → 90° → NA



## Quantities to analyze


### 1. Colour performance

Rosch–MacAdam colour solid

Optimal colour contour

CIELAB a*–b*

Colour strips

η_C, η_L heatmaps

### 2. Scattering

Differential scattering vs θ (dB)

Integrated efficiencies vs λ

Asymmetry g, albedo

Forward / side / backward (dB)

Pie chart at α = 70.5°

### 3. Near fields

|E| (dB), xz / yz / xy

Poynting flow + streamlines

Hotspots

### 4. Resonances

Mie l‑decomposition

l‑resolved spectra

### 5. Sensitivity

η_C, η_L vs n_layers, Δn, k

In [ ]:
### !!!! Use bragg_notebook conda environment. However it does not contain scattnay !!!!
# CELL 0 — IMPORTS & GLOBAL SETTINGSCELL 0 — IMPORTS & GLOBAL SETTINGS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from bragg_onion.colour_solid_plotting import (
    load_rosch_macadam_colour_solid,
    plot_rosch_macadam_colour_solid,
    add_cielab_point,
)

import importlib.util

OUTPUT_DIR = Path.cwd() / "outputs" / "analyse_constant_index_study_detailed_coarse"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HAS_PYARROW = importlib.util.find_spec("pyarrow") is not None
HAS_FASTPARQUET = importlib.util.find_spec("fastparquet") is not None

USE_PARQUET = HAS_PYARROW or HAS_FASTPARQUET

print("Parquet support available:", USE_PARQUET)
print("Using format:", "parquet" if USE_PARQUET else "pickle")


In [ ]:
# CELL 1 — PARAMETER SPACE DEFINITION
n_A_vals = np.linspace(1.5, 3.0, 8)
k_vals   = np.linspace(0.0, 0.2, 7)
n_B = 1.5

design_wavelengths_nm = np.arange(400, 781, 20)
n_layers_vals = [10, 20, 30, 40, 50, 60, 80, 100]
half_angles_deg = np.array([10, 30, 50, 70.5, 90])
collection_NA_vals = np.sin(np.deg2rad(half_angles_deg))

In [ ]:
# CELL 2 — MASTER PARAMETER GRID (PRIMARY SWEEP)
# This sweep is colour + scattering only (no fields).
PARAMETER_GRID = {
    "peak_wavelength_m": design_wavelengths_nm * 1e-9,
    "n_layers": n_layers_vals,
    "outer_layer": ["A"],
    "core_thickness_factor": 0.5,
}

In [ ]:
# !!!!!!RUN run_primary_sweep.py on THE WSL THREADRIPPER, with the Windows D: drive as output.!!!!!!!!

In [ ]:
# Cell 4 — LOAD COPIED WORKSTATION OUTPUTS (combined file preferred, else combine chunks)
from pathlib import Path
import pandas as pd
import importlib.util

# ------------------------------------------------------------
# Path on YOUR LAPTOP where you copied the workstation results
# ------------------------------------------------------------
SOURCE_OUTPUT_DIR = Path(
    r"C:\Users\SchwarzN\OneDrive - Université de Fribourg\Institution\P1_BraggSpericalPigments\Simulation\MIE_Computation\Scattnlay_BraggOnion\outputs\analyse_constant_index_study_detailed_coarse"
)

CHECKPOINT_DIR = SOURCE_OUTPUT_DIR / "primary_sweep_checkpoints"

# ------------------------------------------------------------
# Check parquet support on THIS machine
# ------------------------------------------------------------
HAS_PYARROW = importlib.util.find_spec("pyarrow") is not None
HAS_FASTPARQUET = importlib.util.find_spec("fastparquet") is not None
HAS_PARQUET_SUPPORT = HAS_PYARROW or HAS_FASTPARQUET

print("SOURCE_OUTPUT_DIR:", SOURCE_OUTPUT_DIR)
print("CHECKPOINT_DIR    :", CHECKPOINT_DIR)
print("Parquet support   :", HAS_PARQUET_SUPPORT)

# ------------------------------------------------------------
# Inspect what files actually exist
# ------------------------------------------------------------
combined_parquet = SOURCE_OUTPUT_DIR / "df_all.parquet"
combined_pickle  = SOURCE_OUTPUT_DIR / "df_all.pkl"

parquet_parts = sorted(CHECKPOINT_DIR.glob("chunk_*.parquet"))
pickle_parts  = sorted(CHECKPOINT_DIR.glob("chunk_*.pkl"))

print("combined_parquet exists:", combined_parquet.exists())
print("combined_pickle  exists:", combined_pickle.exists())
print("number of parquet chunks:", len(parquet_parts))
print("number of pickle  chunks:", len(pickle_parts))

# Optional: show a few files for debugging
if SOURCE_OUTPUT_DIR.exists():
    print("\nTop-level files in SOURCE_OUTPUT_DIR:")
    for p in sorted(SOURCE_OUTPUT_DIR.iterdir())[:20]:
        print("  ", p.name)

if CHECKPOINT_DIR.exists():
    print("\nFirst files in CHECKPOINT_DIR:")
    for p in sorted(CHECKPOINT_DIR.iterdir())[:10]:
        print("  ", p.name)

# ------------------------------------------------------------
# Load logic
# Priority:
# 1) existing combined parquet
# 2) existing combined pickle
# 3) combine parquet chunks
# 4) combine pickle chunks
# ------------------------------------------------------------
if combined_parquet.exists():
    if not HAS_PARQUET_SUPPORT:
        raise ImportError(
            f"Found {combined_parquet}, but pyarrow/fastparquet is not installed on this laptop. "
            "Install pyarrow with: python -m pip install pyarrow"
        )
    df_all = pd.read_parquet(combined_parquet)
    USED_FORMAT = "parquet"
    LOADED_FROM = combined_parquet

elif combined_pickle.exists():
    df_all = pd.read_pickle(combined_pickle)
    USED_FORMAT = "pickle"
    LOADED_FROM = combined_pickle

elif parquet_parts:
    if not HAS_PARQUET_SUPPORT:
        raise ImportError(
            f"Found parquet chunk files in {CHECKPOINT_DIR}, but pyarrow/fastparquet is not installed on this laptop. "
            "Install pyarrow with: python -m pip install pyarrow"
        )
    df_all = pd.concat([pd.read_parquet(p) for p in parquet_parts], ignore_index=True)
    USED_FORMAT = "parquet"
    LOADED_FROM = CHECKPOINT_DIR

    # save combined once
    df_all.to_parquet(combined_parquet, index=False)
    print(f"\nCombined dataframe saved to: {combined_parquet}")

elif pickle_parts:
    df_all = pd.concat([pd.read_pickle(p) for p in pickle_parts], ignore_index=True)
    USED_FORMAT = "pickle"
    LOADED_FROM = CHECKPOINT_DIR

    # save combined once
    df_all.to_pickle(combined_pickle)
    print(f"\nCombined dataframe saved to: {combined_pickle}")

else:
    raise FileNotFoundError(
        f"No combined dataframe and no checkpoint chunk files were found.\n"
        f"Checked:\n"
        f"  - {combined_parquet}\n"
        f"  - {combined_pickle}\n"
        f"  - {CHECKPOINT_DIR / 'chunk_*.parquet'}\n"
        f"  - {CHECKPOINT_DIR / 'chunk_*.pkl'}"
    )

print(f"\nLoaded df_all from: {LOADED_FROM}")
print(f"Used format       : {USED_FORMAT}")
print(f"Total rows in df_all: {len(df_all)}")

# ------------------------------------------------------------
# Make the rest of the notebook use this output directory
# ------------------------------------------------------------
OUTPUT_DIR = SOURCE_OUTPUT_DIR
USE_PARQUET = (USED_FORMAT == "parquet")

In [ ]:
# CELL 5 — Rosch–MacAdam colour solid
# Supports raw / display / both
# Marker symbol chosen by user-selected variable
# Marker edge colour configurable
# Filled-symbol-aware styling
# Loads optimal-colour contour from the bragg_onion package directory

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import bragg_onion

# ------------------------------------------------------------
# 0) Choose dataframe
# ------------------------------------------------------------
if "df_plot" not in globals():
    if "df_all" not in globals():
        raise NameError("Neither df_plot nor df_all is available.")
    df_plot = df_all.copy()

# ------------------------------------------------------------
# 1) User options
# ------------------------------------------------------------
COLOUR_VIEW = "both"   # choose: "raw", "display", or "both"

MAX_OVERLAY_POINTS = 2000
FIXED_MARKER_SIZE = 4
SHOW_FIGURES = True
SAVE_HTML = True

# Choose WHICH variable controls marker symbol
# Examples:
# "n_layers", "collection_angle_deg", "n_A", "k", "outer_layer"
MARKER_SYMBOL_VARIABLE = "n_layers"

# Choose symbol family:
# "solid_only"      -> circle, square, diamond
# "solid_and_line"  -> circle, square, diamond, cross, x
MARKER_SYMBOL_STYLE = "solid_and_line"

# Marker edge colour:
# choose "black" for black outline
# choose None for no outline
MARKER_EDGE_COLOR = "black"

# Marker edge width (used only for filled markers if edge is enabled)
MARKER_EDGE_WIDTH = 0.4

# Optimal-colour marker styling
OPTIMAL_MARKER_SIZE = 4.8
OPTIMAL_MARKER_EDGE_COLOR = None
OPTIMAL_MARKER_EDGE_WIDTH = 0.7

# ------------------------------------------------------------
# 2) Basic checks
# ------------------------------------------------------------
if MARKER_SYMBOL_VARIABLE not in df_plot.columns:
    raise KeyError(
        f"MARKER_SYMBOL_VARIABLE='{MARKER_SYMBOL_VARIABLE}' is not a column in df_plot.\n"
        f"Available columns include:\n{list(df_plot.columns)}"
    )

# ------------------------------------------------------------
# 3) Symbol families
# ------------------------------------------------------------
FILLED_SYMBOLS = ["circle", "square", "diamond"]
LINE_SYMBOLS = ["cross", "x"]

if MARKER_SYMBOL_STYLE == "solid_only":
    valid_symbols = FILLED_SYMBOLS
elif MARKER_SYMBOL_STYLE == "solid_and_line":
    valid_symbols = FILLED_SYMBOLS + LINE_SYMBOLS
else:
    raise ValueError("MARKER_SYMBOL_STYLE must be 'solid_only' or 'solid_and_line'.")

# Create stable grouping values
symbol_values = df_plot[MARKER_SYMBOL_VARIABLE].dropna().unique().tolist()

# Sort values if possible
try:
    symbol_values = sorted(symbol_values)
except Exception:
    symbol_values = list(symbol_values)

symbol_map = {
    val: valid_symbols[i % len(valid_symbols)]
    for i, val in enumerate(symbol_values)
}

print(f"{MARKER_SYMBOL_VARIABLE} → marker symbol mapping:")
for val, sym in symbol_map.items():
    print(f"  {val}: {sym}")

# ------------------------------------------------------------
# 4) Optional overlay reduction
# ------------------------------------------------------------
if len(df_plot) > MAX_OVERLAY_POINTS:
    df_overlay = (
        df_plot.sort_values(["eta_C", "eta_L", "eta_Y"], ascending=False)
        .head(MAX_OVERLAY_POINTS)
        .copy()
    )
    print(f"Overlaying top {MAX_OVERLAY_POINTS} cases.")
else:
    df_overlay = df_plot.copy()

# ------------------------------------------------------------
# 5) Load Rosch–MacAdam solid
# ------------------------------------------------------------
solid_df = load_rosch_macadam_colour_solid()

# ------------------------------------------------------------
# 6) Load optimal-colour contour from bragg_onion package directory
# ------------------------------------------------------------
opt_csv_path = Path(bragg_onion.__file__).resolve().parent / "rosch_macadam_max_chroma_per_hue_1deg.csv"
if not opt_csv_path.exists():
    raise FileNotFoundError(f"Optimal-colour CSV not found at: {opt_csv_path}")

opt_df = pd.read_csv(opt_csv_path)

L_opt_col = "L_smooth" if "L_smooth" in opt_df.columns else "L"
a_opt_col = "a_smooth" if "a_smooth" in opt_df.columns else "a"
b_opt_col = "b_smooth" if "b_smooth" in opt_df.columns else "b"
hex_opt_col = "hex_smooth" if "hex_smooth" in opt_df.columns else None

print(f"Loaded optimal-colour CSV from: {opt_csv_path}")

# ------------------------------------------------------------
# 7) Helper: choose columns for a given colour mode
# ------------------------------------------------------------
def choose_colour_columns(mode: str):
    if mode == "display":
        required_cols = ["L_display", "a_display", "b_display", "hex_display"]
        missing = [c for c in required_cols if c not in df_overlay.columns]
        if missing:
            raise KeyError(
                f"Missing display-normalized columns: {missing}. "
                "Run the display-normalized colour cell first."
            )
        return {
            "L_col": "L_display",
            "a_col": "a_display",
            "b_col": "b_display",
            "hex_col": "hex_display",
            "title_suffix": "display-normalized",
        }

    elif mode == "raw":
        required_cols = ["L", "a", "b", "hex"]
        missing = [c for c in required_cols if c not in df_overlay.columns]
        if missing:
            raise KeyError(f"Missing raw colour columns: {missing}")
        return {
            "L_col": "L",
            "a_col": "a",
            "b_col": "b",
            "hex_col": "hex",
            "title_suffix": "raw",
        }

    else:
        raise ValueError("mode must be 'raw' or 'display'.")

# ------------------------------------------------------------
# 8) Helper: marker line style
#     Edge styling only affects FILLED symbols.
#     For line-only symbols (cross, x), line width is forced to 0.
# ------------------------------------------------------------
def marker_line_kwargs(symbol, edge_color, edge_width):
    if symbol in FILLED_SYMBOLS:
        if edge_color is None:
            return dict(width=0)
        return dict(color=edge_color, width=edge_width)
    else:
        return dict(width=0)

# ------------------------------------------------------------
# 9) Helper: build one figure
# ------------------------------------------------------------
def build_rosch_macadam_figure(mode: str):
    cols = choose_colour_columns(mode)
    L_col = cols["L_col"]
    a_col = cols["a_col"]
    b_col = cols["b_col"]
    hex_col = cols["hex_col"]
    title_suffix = cols["title_suffix"]

    fig = plot_rosch_macadam_colour_solid(
        solid_df,
        title=f"Bragg-onion cases in Rosch–MacAdam colour solid ({title_suffix})",
        max_points=50000,
        marker_size=1.2,
        opacity=0.05,
    )

    # --------------------------------------------------------
    # Optimal-colour contour line
    # --------------------------------------------------------
    fig.add_trace(
        go.Scatter3d(
            x=opt_df[a_opt_col],
            y=opt_df[b_opt_col],
            z=opt_df[L_opt_col],
            mode="lines",
            line=dict(color="black", width=5),
            name="optimal colour contour",
            hoverinfo="skip",
        )
    )

    # --------------------------------------------------------
    # Optimal-colour filled markers
    # --------------------------------------------------------
    hover_opt = [
        f"hue={row['hue_deg']:.1f}°<br>"
        f"L={row[L_opt_col]:.2f}, a={row[a_opt_col]:.2f}, b={row[b_opt_col]:.2f}"
        for _, row in opt_df.iterrows()
    ]

    fig.add_trace(
        go.Scatter3d(
            x=opt_df[a_opt_col],
            y=opt_df[b_opt_col],
            z=opt_df[L_opt_col],
            mode="markers",
            marker=dict(
                size=OPTIMAL_MARKER_SIZE,
                color=opt_df[hex_opt_col] if hex_opt_col is not None else "black",
                symbol="circle",
                line=marker_line_kwargs("circle", OPTIMAL_MARKER_EDGE_COLOR, OPTIMAL_MARKER_EDGE_WIDTH),
                opacity=1.0,
            ),
            name="optimal colours",
            hovertext=hover_opt,
            hoverinfo="text",
        )
    )

    # --------------------------------------------------------
    # Group Bragg-onion cases by chosen marker variable
    # --------------------------------------------------------
    for group_value, grp in df_overlay.groupby(MARKER_SYMBOL_VARIABLE, sort=False):
        point_symbol = symbol_map.get(group_value, "circle")

        hovertext = []
        for _, row in grp.iterrows():
            lines = [
                f"case_index: {int(row['case_index'])}",
                f"design λ: {row['design_peak_wavelength_nm']:.1f} nm"
                if "design_peak_wavelength_nm" in grp.columns else None,
                f"{MARKER_SYMBOL_VARIABLE}: {row[MARKER_SYMBOL_VARIABLE]}",
                f"n_layers: {int(row['n_layers'])}" if "n_layers" in grp.columns else None,
                f"outer_layer: {row['outer_layer']}" if "outer_layer" in grp.columns else None,
                f"n_A: {row['n_A']:.3f}" if "n_A" in grp.columns else None,
                f"k: {row['k']:.3f}" if "k" in grp.columns else None,
                f"collection angle: {row['collection_angle_deg']:.1f}°"
                if "collection_angle_deg" in grp.columns else None,
                f"eta_C: {row['eta_C']:.3f}" if "eta_C" in grp.columns else None,
                f"eta_L: {row['eta_L']:.3f}" if "eta_L" in grp.columns else None,
                f"eta_Y: {row['eta_Y']:.3f}" if "eta_Y" in grp.columns else None,
                f"q_collected_geom_max: {row['q_collected_geom_max']:.3f}"
                if "q_collected_geom_max" in grp.columns else None,
                f"hex: {row[hex_col]}",
            ]
            lines = [line for line in lines if line is not None]
            hovertext.append("<br>".join(lines))

        fig.add_trace(
            go.Scatter3d(
                x=grp[a_col],
                y=grp[b_col],
                z=grp[L_col],
                mode="markers",
                marker=dict(
                    size=FIXED_MARKER_SIZE,
                    color=grp[hex_col],
                    symbol=point_symbol,
                    line=marker_line_kwargs(point_symbol, MARKER_EDGE_COLOR, MARKER_EDGE_WIDTH),
                    opacity=0.95,
                ),
                name=f"{MARKER_SYMBOL_VARIABLE} = {group_value}",
                hovertext=hovertext,
                hoverinfo="text",
            )
        )

    fig.update_layout(
        scene=dict(
            xaxis=dict(title="a*", tickformat=".1f"),
            yaxis=dict(title="b*", tickformat=".1f"),
            zaxis=dict(title="L*", tickformat=".1f"),
            aspectmode="cube",
            camera=dict(eye=dict(x=1.8, y=1.8, z=1.2)),
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        legend=dict(itemsizing="constant"),
    )

    return fig, title_suffix

# ------------------------------------------------------------
# 10) Render according to mode
# ------------------------------------------------------------
if COLOUR_VIEW == "both":
    modes_to_plot = ["raw", "display"]
else:
    modes_to_plot = [COLOUR_VIEW]

generated_figures = {}

for mode in modes_to_plot:
    fig, suffix = build_rosch_macadam_figure(mode)
    generated_figures[mode] = fig

    if SAVE_HTML:
        html_path = OUTPUT_DIR / f"rosch_macadam_colour_solid_{suffix}_with_optimal_colours.html"
        fig.write_html(str(html_path))
        print(f"Saved interactive 3D figure to: {html_path}")

    if SHOW_FIGURES:
        fig.show()

In [ ]:
# (1.2) CIELAB a*–b* plane
plt.figure(figsize=(6,6))
plt.scatter(df_all["a"], df_all["b"], c=df_all["hex"], s=15)
plt.axhline(0, color="gray")
plt.axvline(0, color="gray")
plt.xlabel("a*")
plt.ylabel("b*")
plt.title("CIELAB a*–b* plane")
plt.axis("equal")
plt.show()

In [ ]:
# (1.3) Colour strips (λ‑dependence)
from bragg_onion.sweep import plot_sweep_colour_strip

plot_sweep_colour_strip(
    df_all.sort_values("design_peak_wavelength_nm"),
    x="design_peak_wavelength_nm",
    colour_hex_col="hex",
    label_col="n_layers",
)
plt.show()

CELL 6 — η_C AND η_L HEATMAPS (1)

In [ ]:
subset = df_all[
    (df_all["n_A"] == 2.2) &
    (df_all["k"] == 0.05) &
    (np.isclose(df_all["collection_angle_deg"], 70.5))
]

from bragg_onion.sweep import plot_sweep_heatmap

plot_sweep_heatmap(
    subset,
    x="design_peak_wavelength_nm",
    y="n_layers",
    value="eta_C",
)
plt.title("η_C heatmap")
plt.show()

plot_sweep_heatmap(
    subset,
    x="design_peak_wavelength_nm",
    y="n_layers",
    value="eta_L",
)
plt.title("η_L heatmap")
plt.show()

CELL 7 — SCATTERING ANALYSIS (2.0)

In [ ]:
# (2.0.1) Differential scattering (dB)
case = sweep_result.case_results[0]

from bragg_onion.scattering_analysis import plot_differential_scattering_vs_angle

plot_differential_scattering_vs_angle(
    case.scattering_result,
    normalization="geom",
    as_db=True,
)
plt.show()

In [ ]:
# (2.0.2) Integrated scattering vs wavelength
wl_nm = case.scattering_result.wavelengths_m * 1e9

plt.plot(wl_nm, case.scattering_result.qsca, label="Qsca")
plt.plot(wl_nm, case.scattering_result.qext, label="Qext")
plt.legend()
plt.xlabel("Wavelength [nm]")
plt.ylabel("Efficiency")
plt.show()

In [ ]:
# (2.0.3) g, albedo, forward/side/backward (dB)
from bragg_onion.scattering_analysis import plot_forward_side_backward_vs_wavelength

plot_forward_side_backward_vs_wavelength(
    case.scattering_result,
    alpha_deg=70.5,
    normalize="geom",
    as_db=True,
)
plt.show()

In [ ]:
# (2.0.4) Pie chart
from bragg_onion.scattering_analysis import plot_partition_pie_at_wavelength

plot_partition_pie_at_wavelength(
    case.scattering_result,
    alpha_deg=70.5,
)
plt.show()

CELL 8 — NEAR‑FIELD ANALYSIS (2.1)

In [ ]:
# For selected/top cases only:
from bragg_onion.fields import compute_field_map, plot_field_magnitude, plot_poynting_streamlines
field_map = compute_field_map(
    stack=stack_sel,
    wavelength_m=wl_res_nm * 1e-9,
    n_medium=1.0,
    plane="xz",
    npts=401,
    extent_outer_radius_factor=3.0,
)

plot_field_magnitude(
    field_map,
    quantity="E",
    kind="total",
    scale="db",
)
plt.show()

plot_poynting_streamlines(
    field_map,
    flow_kind="delta",
    background_quantity="S",
    background_kind="delta",
    background_scale="db",
)
plt.show()

######Repeat for yz and xy.###########

CELL 9 — MIE RESONANCES (2.2)

You already implemented this in Point 6:

- l‑resolved bar plots

- l‑resolved spectra

✅ reuse those exact cells here for:

- top cases

- representative parameter cuts

In [ ]:
# CELL 10 — SENSITIVITY ANALYSIS (OPTIONAL, RECOMMENDED)
# Example: η_C vs n_layers for different n_A.
plt.figure(figsize=(7,4.5))

for n_A in n_A_vals:
    subset = df_all[
        (df_all["n_A"] == n_A) &
        (df_all["k"] == 0.05) &
        (np.isclose(df_all["collection_angle_deg"], 70.5))
    ]
    grp = subset.groupby("n_layers")["eta_C"].mean()
    plt.plot(grp.index, grp.values, marker="o", label=f"n_A={n_A:.2f}")

plt.xlabel("Number of layers")
plt.ylabel("η_C")
plt.legend()
plt.title("Sensitivity of η_C to n_layers and refractive index contrast")
plt.show()